# HIV Activity Predictor — Demo

This lightweight notebook shows how to use the **refactored Python modules**
(`src.inference`) to load pre-trained checkpoints and visualise predictions.

**Prerequisites:**
1. Install dependencies: `pip install -r requirements.txt`
2. Train at least one model: `python -m src.train --model random_forest --epochs 30`

No training code lives in this notebook — all heavy lifting is in `src/`.

In [ ]:
import sys, os
# Ensure repo root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

print('Environment ready.')

## 1. Define test molecules

We use a handful of well-known compounds as examples.

In [ ]:
# Example SMILES: a few molecules from the HIV dataset and known drugs
SMILES_EXAMPLES = [
    'CCO',                              # ethanol (trivial, likely inactive)
    'c1ccccc1',                         # benzene
    'CC(=O)Nc1ccc(O)cc1',              # paracetamol
    'OC[C@H]1O[C@@H](n2cnc3c(N)ncnc23)[C@H](O)[C@@H]1O',  # adenosine
    'CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C',  # testosterone-like
]

print(f'{len(SMILES_EXAMPLES)} molecules loaded.')

## 2. Run inference

We import `predict_smiles` from `src.inference` and run predictions with the
Random Forest model (fastest, no GPU required).

In [ ]:
from src.inference import predict_smiles

MODEL_NAME = 'random_forest'   # change to 'graphconv' or 'attentivefp' if available
MODEL_DIR  = '../models'        # adjust path if needed

try:
    results = predict_smiles(
        smiles_list=SMILES_EXAMPLES,
        model_name=MODEL_NAME,
        threshold=0.5,
        model_dir=MODEL_DIR,
    )
    print('Prediction succeeded!')
except FileNotFoundError as e:
    print(f'Checkpoint not found: {e}')
    print(f'Train first: python -m src.train --model {MODEL_NAME} --epochs 30')
    results = None

## 3. Display results as a table

In [ ]:
import pandas as pd

if results is not None:
    df = pd.DataFrame({
        'SMILES':      results['smiles'],
        'Probability': results['predictions'],
        'Label':       ['ACTIVE' if l == 1 else 'inactive' for l in results['labels']],
    })
    display(df.style.format({'Probability': '{:.4f}'}))
else:
    print('No results to display — train the model first.')

## 4. Visualise probability distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if results is not None:
    fig, ax = plt.subplots(figsize=(8, 4))

    colors = ['#E24B4A' if l == 1 else '#378ADD' for l in results['labels']]
    bars = ax.barh(range(len(results['smiles'])), results['predictions'],
                   color=colors, edgecolor='white')

    ax.axvline(0.5, color='black', linestyle='--', linewidth=1, label='threshold=0.5')
    ax.set_yticks(range(len(results['smiles'])))
    ax.set_yticklabels(
        [s[:30] + '…' if len(s) > 30 else s for s in results['smiles']],
        fontsize=9
    )
    ax.set_xlabel('Predicted probability of anti-HIV activity')
    ax.set_title(f'HIV Activity Predictions — {MODEL_NAME}')
    ax.legend()

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#E24B4A', label='Predicted ACTIVE'),
        Patch(facecolor='#378ADD', label='Predicted inactive'),
    ]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.show()
else:
    print('No results to visualise.')

## 5. Multi-model comparison (optional)

Run this cell only if all three models have been trained.

In [ ]:
from src.inference import predict_smiles

all_results = {}
for model_name in ['random_forest', 'graphconv', 'attentivefp']:
    try:
        res = predict_smiles(SMILES_EXAMPLES, model_name, model_dir=MODEL_DIR)
        all_results[model_name] = res['predictions']
        print(f'  {model_name}: OK')
    except FileNotFoundError:
        print(f'  {model_name}: checkpoint missing — skipped')

if all_results:
    comparison_df = pd.DataFrame(all_results, index=results['smiles'])
    display(comparison_df.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=None))